<a href="https://colab.research.google.com/github/Idontnol/studentPerformance/blob/main/.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate pillow huggingface_hub

In [2]:
import zipfile
from pathlib import Path

zip_path = "/content/screenshots-grc.zip"
extract_dir = Path("/content/my_images")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

image_files = sorted(
    [p for p in extract_dir.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}]
)
print(f"Found {len(image_files)} images2.")

Found 288 images2.


In [8]:
import json
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "ibm-granite/granite-vision-4.1-4b"
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda",
).eval()

prompt = "Extract all visible text from this image accurately, preserving line breaks and structure."
results = []

for idx, img_path in enumerate(image_files, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        formatted_prompt = processor.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[formatted_prompt],
            images=[image],
            return_tensors="pt",
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2048,
                use_cache=True,
            )

        generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
        text = processor.decode(generated_tokens, skip_special_tokens=True).strip()
        results.append({"file": img_path.name, "text": text})
        print(f"[{idx}/{len(image_files)}] Done: {img_path.name}")
    except Exception as e:
        results.append({"file": img_path.name, "error": str(e)})
        print(f"[{idx}/{len(image_files)}] Failed {img_path.name}: {e}")

with open("/content/extracted_texts.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

with open("/content/extracted_texts.txt", "w", encoding="utf-8") as f:
    for item in results:
        f.write(f"=== {item['file']} ===\n")
        f.write(item.get("text") or f"ERROR: {item.get('error', '')}")
        f.write("\n\n")

print("Saved /content/extracted_texts.txt and /content/extracted_texts.json")

processor_config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/1083 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[1/288] Done: CIS_GRC_Question_1.png
[2/288] Done: CIS_GRC_Question_10.png
[3/288] Done: CIS_GRC_Question_100.png
[4/288] Done: CIS_GRC_Question_101.png
[5/288] Done: CIS_GRC_Question_102.png
[6/288] Done: CIS_GRC_Question_103.png
[7/288] Done: CIS_GRC_Question_104.png
[8/288] Done: CIS_GRC_Question_105.png
[9/288] Done: CIS_GRC_Question_106.png
[10/288] Done: CIS_GRC_Question_107.png
[11/288] Done: CIS_GRC_Question_108.png
[12/288] Done: CIS_GRC_Question_109.png
[13/288] Done: CIS_GRC_Question_11.png
[14/288] Done: CIS_GRC_Question_110.png


KeyboardInterrupt: 

In [3]:
import gc
import json
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

# Clear previous model and free GPU memory to prevent Out-Of-Memory errors
if "model" in globals():
    del model
if "processor" in globals():
    del processor

gc.collect()
torch.cuda.empty_cache()

model_id = "ibm-granite/granite-vision-4.1-4b"
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="auto",
).eval()



Loading weights:   0%|          | 0/1083 [00:00<?, ?it/s]

In [4]:
prompt = "Extract all visible text from this image accurately, preserving line breaks and structure."
results = []

for idx, img_path in enumerate(image_files, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        formatted_prompt = processor.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[formatted_prompt],
            images=[image],
            return_tensors="pt",
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                use_cache=True,
            )

        generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
        text = processor.decode(generated_tokens, skip_special_tokens=True).strip()
        results.append({"file": img_path.name, "text": text})
        print(f"[{idx}/{len(image_files)}] Done: {img_path.name}")
    except Exception as e:
        results.append({"file": img_path.name, "error": str(e)})
        print(f"[{idx}/{len(image_files)}] Failed {img_path.name}: {e}")

    with open("/content/extracted_texts.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

with open("/content/extracted_texts.txt", "w", encoding="utf-8") as f:
    for item in results:
        f.write(f"=== {item['file']} ===\n")
        f.write(item.get("text") or f"ERROR: {item.get('error', '')}")
        f.write("\n\n")

print("Saved /content/extracted_texts.txt and /content/extracted_texts.json")

[1/288] Done: CIS_GRC_Question_1.png
[2/288] Done: CIS_GRC_Question_10.png
[3/288] Done: CIS_GRC_Question_100.png
[4/288] Done: CIS_GRC_Question_101.png
[5/288] Done: CIS_GRC_Question_102.png
[6/288] Done: CIS_GRC_Question_103.png
[7/288] Done: CIS_GRC_Question_104.png
[8/288] Done: CIS_GRC_Question_105.png
[9/288] Done: CIS_GRC_Question_106.png
[10/288] Done: CIS_GRC_Question_107.png
[11/288] Done: CIS_GRC_Question_108.png
[12/288] Done: CIS_GRC_Question_109.png
[13/288] Done: CIS_GRC_Question_11.png
[14/288] Done: CIS_GRC_Question_110.png
[15/288] Done: CIS_GRC_Question_111.png


KeyboardInterrupt: 

In [5]:
!sudo apt-get update -qq && sudo apt-get install -y -qq tesseract-ocr libtesseract-dev
!pip install -q pytesseract pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 10.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libattr1_1%3a2.5.2-1ubuntu0.1_amd64.deb ...
Unpacking libattr1:amd64 (1:2.5.2-1ubuntu0.1) over (1:2.5.2-1build1.1) ...
Setting up libattr1:amd64 (1:2.5.2-1ubuntu0.1) ...
Selecting previously unselected package libattr1-dev:amd64.
(Reading database ... 127324 

In [6]:
import json
from pathlib import Path
import pytesseract
from PIL import Image

image_dir = Path("/content/my_images")
image_files = sorted(
    [
        p
        for p in image_dir.rglob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
    ]
)

output_json = Path("/content/extracted_texts.json")
output_txt = Path("/content/extracted_texts.txt")

# Resume from existing JSON if present
results = []
done_files = set()
if output_json.exists():
    results = json.loads(output_json.read_text(encoding="utf-8"))
    done_files = {item["file"] for item in results if "text" in item}

# --psm 6 assumes a single uniform block of text; -l eng sets English
custom_config = r"--psm 6 -l eng"

for idx, img_path in enumerate(image_files, start=1):
    if img_path.name in done_files:
        print(f"[{idx}/{len(image_files)}] Skipping (already done): {img_path.name}")
        continue

    try:
        image = Image.open(img_path).convert("RGB")
        text = pytesseract.image_to_string(image, config=custom_config).strip()
        results.append({"file": img_path.name, "text": text})
        print(f"[{idx}/{len(image_files)}] Done: {img_path.name}")
    except Exception as e:
        results.append({"file": img_path.name, "error": str(e)})
        print(f"[{idx}/{len(image_files)}] Failed {img_path.name}: {e}")

    # Save checkpoint after each image
    output_json.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

with open(output_txt, "w", encoding="utf-8") as f:
    for item in results:
        f.write(f"=== {item['file']} ===\n")
        f.write(item.get("text") or f"ERROR: {item.get('error', '')}")
        f.write("\n\n")

print(f"Processed {len(results)} items. Saved {output_txt} and {output_json}")

[1/288] Skipping (already done): CIS_GRC_Question_1.png
[2/288] Skipping (already done): CIS_GRC_Question_10.png
[3/288] Skipping (already done): CIS_GRC_Question_100.png
[4/288] Skipping (already done): CIS_GRC_Question_101.png
[5/288] Skipping (already done): CIS_GRC_Question_102.png
[6/288] Skipping (already done): CIS_GRC_Question_103.png
[7/288] Skipping (already done): CIS_GRC_Question_104.png
[8/288] Skipping (already done): CIS_GRC_Question_105.png
[9/288] Skipping (already done): CIS_GRC_Question_106.png
[10/288] Skipping (already done): CIS_GRC_Question_107.png
[11/288] Skipping (already done): CIS_GRC_Question_108.png
[12/288] Skipping (already done): CIS_GRC_Question_109.png
[13/288] Skipping (already done): CIS_GRC_Question_11.png
[14/288] Skipping (already done): CIS_GRC_Question_110.png
[15/288] Skipping (already done): CIS_GRC_Question_111.png
[16/288] Done: CIS_GRC_Question_112.png
[17/288] Done: CIS_GRC_Question_113.png
[18/288] Done: CIS_GRC_Question_114.png
[19/288]